# Generate LaTeX Tables for Research Paper

This notebook generates LaTeX tables from the aggregated statistics for inclusion in the research paper.

In [1]:
import json
from pathlib import Path

import pandas as pd

output_dir = Path("../output")
stats_file = output_dir / "aggregated_statistics.json"

# Load aggregated statistics
with open(stats_file) as f:
    stats_by_language = json.load(f)

print(f"Loaded statistics for {len(stats_by_language)} languages")
print(f"Languages: {list(stats_by_language.keys())}")

Loaded statistics for 5 languages
Languages: ['csharp', 'go', 'python', 'rust', 'typescript']


In [2]:
# Combined General + RQ1 + RQ2 Table
# Languages across the top, statistics down the side

# Order by number of repositories (descending)
langs = ["typescript", "python", "go", "csharp", "rust"]
lang_labels = [f"\\{lang}" for lang in langs]  # Use LaTeX macro format


# Helper functions for formatting
def format_count_pct(count, pct):
    """Format as 'count (pct%)'."""
    return f"{count:,} ({pct:.1f}\\%)"


def format_avg(avg):
    """Format average to 2 decimal places."""
    return f"{avg:.2f}"


# Calculate totals across all languages
total_repos = sum(stats_by_language[lang]["num_repos"] for lang in langs)
total_prs = sum(stats_by_language[lang]["total_prs"] for lang in langs)

# Sum counts for aggregation
total_prs_with_any_import = sum(
    stats_by_language[lang]["prs_with_any_import"] for lang in langs
)
total_prs_with_stdlib = sum(
    stats_by_language[lang]["prs_with_stdlib"] for lang in langs
)
total_prs_with_extlib = sum(
    stats_by_language[lang]["prs_with_extlib"] for lang in langs
)
total_prs_with_new_deps = sum(
    stats_by_language[lang]["prs_with_new_deps"] for lang in langs
)

# Sum totals for weighted averages
total_stdlib_imports_count = sum(
    stats_by_language[lang]["avg_stdlib_per_pr"] * stats_by_language[lang]["total_prs"]
    for lang in langs
)
total_extlib_imports_count = sum(
    stats_by_language[lang]["avg_extlib_per_pr"] * stats_by_language[lang]["total_prs"]
    for lang in langs
)

# Sum dependency counts
total_all_new_deps = sum(stats_by_language[lang]["total_new_deps"] for lang in langs)
total_new_deps_with_version = sum(
    stats_by_language[lang]["total_new_deps_with_version"] for lang in langs
)
total_unique_deps = sum(
    stats_by_language[lang]["total_unique_new_deps"] for lang in langs
)

# Sum unique library counts
total_unique_extlib = sum(
    stats_by_language[lang]["total_unique_extlib_imports"] for lang in langs
)

# Calculate aggregated percentages
pct_any_import_all = 100 * total_prs_with_any_import / total_prs if total_prs > 0 else 0
pct_stdlib_all = 100 * total_prs_with_stdlib / total_prs if total_prs > 0 else 0
pct_extlib_all = 100 * total_prs_with_extlib / total_prs if total_prs > 0 else 0
pct_new_deps_all = 100 * total_prs_with_new_deps / total_prs if total_prs > 0 else 0
avg_stdlib_all = total_stdlib_imports_count / total_prs if total_prs > 0 else 0
avg_extlib_all = total_extlib_imports_count / total_prs if total_prs > 0 else 0
avg_new_deps_all = total_all_new_deps / total_prs if total_prs > 0 else 0
pct_with_version_all = (
    100 * total_new_deps_with_version / total_all_new_deps
    if total_all_new_deps > 0
    else 0
)
pct_unique_all = (
    100 * total_unique_deps / total_all_new_deps if total_all_new_deps > 0 else 0
)

# Build the data structure with formatted values
rows = []

# General Statistics
rows.append(
    {
        "Metric": "Total repositories",
        **{lang: f"{stats_by_language[lang]['num_repos']:,}" for lang in langs},
        "All": f"{total_repos:,}",
    }
)
rows.append(
    {
        "Metric": "Total pull requests",
        **{lang: f"{stats_by_language[lang]['total_prs']:,}" for lang in langs},
        "All": f"{total_prs:,}",
    }
)

# RQ1: Library Usage
rows.append(
    {
        "Metric": "PRs importing any library",
        **{
            lang: format_count_pct(
                stats_by_language[lang]["prs_with_any_import"],
                stats_by_language[lang]["pct_prs_with_any_import"],
            )
            for lang in langs
        },
        "All": format_count_pct(total_prs_with_any_import, pct_any_import_all),
    }
)
rows.append(
    {
        "Metric": "PRs importing standard library",
        **{
            lang: format_count_pct(
                stats_by_language[lang]["prs_with_stdlib"],
                stats_by_language[lang]["pct_prs_with_stdlib"],
            )
            for lang in langs
        },
        "All": format_count_pct(total_prs_with_stdlib, pct_stdlib_all),
    }
)
rows.append(
    {
        "Metric": "PRs importing external libraries",
        **{
            lang: format_count_pct(
                stats_by_language[lang]["prs_with_extlib"],
                stats_by_language[lang]["pct_prs_with_extlib"],
            )
            for lang in langs
        },
        "All": format_count_pct(total_prs_with_extlib, pct_extlib_all),
    }
)
rows.append(
    {
        "Metric": "Avg. standard library imports per PR",
        **{
            lang: format_avg(stats_by_language[lang]["avg_stdlib_per_pr"])
            for lang in langs
        },
        "All": format_avg(avg_stdlib_all),
    }
)
rows.append(
    {
        "Metric": "Avg. external library imports per PR",
        **{
            lang: format_avg(stats_by_language[lang]["avg_extlib_per_pr"])
            for lang in langs
        },
        "All": format_avg(avg_extlib_all),
    }
)
rows.append(
    {
        "Metric": "Unique external libraries imported",
        **{
            lang: f"{stats_by_language[lang]['total_unique_extlib_imports']:,}"
            for lang in langs
        },
        "All": f"{total_unique_extlib:,}",
    }
)

# RQ2: New Dependencies
rows.append(
    {
        "Metric": "PRs adding new dependencies",
        **{
            lang: format_count_pct(
                stats_by_language[lang]["prs_with_new_deps"],
                stats_by_language[lang]["pct_prs_with_new_deps"],
            )
            for lang in langs
        },
        "All": format_count_pct(total_prs_with_new_deps, pct_new_deps_all),
    }
)
rows.append(
    {
        "Metric": "Avg. new dependencies per PR",
        **{
            lang: format_avg(stats_by_language[lang]["avg_new_deps_per_pr"])
            for lang in langs
        },
        "All": format_avg(avg_new_deps_all),
    }
)
rows.append(
    {
        "Metric": "Total new dependencies",
        **{lang: f"{stats_by_language[lang]['total_new_deps']:,}" for lang in langs},
        "All": f"{total_all_new_deps:,}",
    }
)
rows.append(
    {
        "Metric": "Dependencies specifying version",
        **{
            lang: format_count_pct(
                stats_by_language[lang]["total_new_deps_with_version"],
                stats_by_language[lang]["pct_new_deps_with_version"],
            )
            for lang in langs
        },
        "All": format_count_pct(total_new_deps_with_version, pct_with_version_all),
    }
)
rows.append(
    {
        "Metric": "Unique new dependencies",
        **{
            lang: f"{stats_by_language[lang]['total_unique_new_deps']:,}"
            for lang in langs
        },
        "All": f"{total_unique_deps:,}",
    }
)

# Print preview
df_preview = pd.DataFrame(rows)
print("Preview of data:")
print(df_preview.to_string(index=False))

# Generate custom LaTeX with multirow, bold headers, vertical rules, and equal-width columns
latex_lines = []
# First cell has no left border (@{}), rest of table has vertical rules
latex_lines.append("\\begin{tabular}{@{}c|p{5cm}|r|r|r|r|r|r|}")
latex_lines.append("\\cline{2-8}")  # Line from column 2 to 8, skipping first cell

# Header row - bold, center-aligned using \multicolumn
header_parts = [" ", "\\textbf{Metric}"]
for label in lang_labels:
    header_parts.append(f"\\multicolumn{{1}}{{c|}}{{\\textbf{{{label}}}}}")
header_parts.append("\\textbf{All languages}")
header = " & ".join(header_parts) + " \\\\"
latex_lines.append(header)
latex_lines.append("\\hline")

# General rows (rows 0-1)
latex_lines.append(
    f"\\multicolumn{{1}}{{|c|}}{{\\multirow{{2}}{{*}}{{\\makecell{{\\textbf{{General}} \\\\ \\textbf{{statistics}}}}}}}} & {rows[0]['Metric']} & {rows[0]['typescript']} & {rows[0]['python']} & {rows[0]['go']} & {rows[0]['csharp']} & {rows[0]['rust']} & {rows[0]['All']} \\\\"
)
latex_lines.append(
    f"\\multicolumn{{1}}{{|c|}}{{}} & {rows[1]['Metric']} & {rows[1]['typescript']} & {rows[1]['python']} & {rows[1]['go']} & {rows[1]['csharp']} & {rows[1]['rust']} & {rows[1]['All']} \\\\"
)
latex_lines.append("\\hline")

# RQ1 rows (rows 2-7) - 6 rows
latex_lines.append(
    f"\\multicolumn{{1}}{{|c|}}{{\\multirow{{6}}{{*}}{{\\makecell{{\\textbf{{RQ1 \\textsc{{Library}}}} \\\\ \\textbf{{\\textsc{{Usage}}}}}}}}}} & {rows[2]['Metric']} & {rows[2]['typescript']} & {rows[2]['python']} & {rows[2]['go']} & {rows[2]['csharp']} & {rows[2]['rust']} & {rows[2]['All']} \\\\"
)
latex_lines.append(
    f"\\multicolumn{{1}}{{|c|}}{{}} & {rows[3]['Metric']} & {rows[3]['typescript']} & {rows[3]['python']} & {rows[3]['go']} & {rows[3]['csharp']} & {rows[3]['rust']} & {rows[3]['All']} \\\\"
)
latex_lines.append(
    f"\\multicolumn{{1}}{{|c|}}{{}} & {rows[4]['Metric']} & {rows[4]['typescript']} & {rows[4]['python']} & {rows[4]['go']} & {rows[4]['csharp']} & {rows[4]['rust']} & {rows[4]['All']} \\\\"
)
latex_lines.append(
    f"\\multicolumn{{1}}{{|c|}}{{}} & {rows[5]['Metric']} & {rows[5]['typescript']} & {rows[5]['python']} & {rows[5]['go']} & {rows[5]['csharp']} & {rows[5]['rust']} & {rows[5]['All']} \\\\"
)
latex_lines.append(
    f"\\multicolumn{{1}}{{|c|}}{{}} & {rows[6]['Metric']} & {rows[6]['typescript']} & {rows[6]['python']} & {rows[6]['go']} & {rows[6]['csharp']} & {rows[6]['rust']} & {rows[6]['All']} \\\\"
)
latex_lines.append(
    f"\\multicolumn{{1}}{{|c|}}{{}} & {rows[7]['Metric']} & {rows[7]['typescript']} & {rows[7]['python']} & {rows[7]['go']} & {rows[7]['csharp']} & {rows[7]['rust']} & {rows[7]['All']} \\\\"
)
latex_lines.append("\\hline")

# RQ2 rows (rows 8-12) - 5 rows
latex_lines.append(
    f"\\multicolumn{{1}}{{|c|}}{{\\multirow{{5}}{{*}}{{\\makecell{{\\textbf{{RQ2 \\textsc{{New}}}} \\\\ \\textbf{{\\textsc{{Dependencies}}}}}}}}}} & {rows[8]['Metric']} & {rows[8]['typescript']} & {rows[8]['python']} & {rows[8]['go']} & {rows[8]['csharp']} & {rows[8]['rust']} & {rows[8]['All']} \\\\"
)
latex_lines.append(
    f"\\multicolumn{{1}}{{|c|}}{{}} & {rows[9]['Metric']} & {rows[9]['typescript']} & {rows[9]['python']} & {rows[9]['go']} & {rows[9]['csharp']} & {rows[9]['rust']} & {rows[9]['All']} \\\\"
)
latex_lines.append(
    f"\\multicolumn{{1}}{{|c|}}{{}} & {rows[10]['Metric']} & {rows[10]['typescript']} & {rows[10]['python']} & {rows[10]['go']} & {rows[10]['csharp']} & {rows[10]['rust']} & {rows[10]['All']} \\\\"
)
latex_lines.append(
    f"\\multicolumn{{1}}{{|c|}}{{}} & {rows[11]['Metric']} & {rows[11]['typescript']} & {rows[11]['python']} & {rows[11]['go']} & {rows[11]['csharp']} & {rows[11]['rust']} & {rows[11]['All']} \\\\"
)
latex_lines.append(
    f"\\multicolumn{{1}}{{|c|}}{{}} & {rows[12]['Metric']} & {rows[12]['typescript']} & {rows[12]['python']} & {rows[12]['go']} & {rows[12]['csharp']} & {rows[12]['rust']} & {rows[12]['All']} \\\\"
)
latex_lines.append("\\hline")

latex_lines.append("\\end{tabular}")

latex_combined = "\n".join(latex_lines)

print("\n" + "=" * 80)
print("LaTeX Table (requires \\usepackage{multirow} and \\usepackage{makecell}):")
print("=" * 80)
print(latex_combined)

Preview of data:
                              Metric     typescript         python             go       csharp         rust            All
                  Total repositories            840            530            242          220          159          1,991
                 Total pull requests          7,480          7,190         10,107        1,983        1,079         27,839
           PRs importing any library 2,993 (40.0\%) 2,720 (37.8\%) 1,273 (12.6\%) 902 (45.5\%) 380 (35.2\%) 8,268 (29.7\%)
      PRs importing standard library   828 (11.1\%) 2,383 (33.1\%) 1,244 (12.3\%) 743 (37.5\%)    0 (0.0\%) 5,198 (18.7\%)
    PRs importing external libraries 2,847 (38.1\%) 1,914 (26.6\%)    324 (3.2\%) 738 (37.2\%) 380 (35.2\%) 6,203 (22.3\%)
Avg. standard library imports per PR           0.26           1.28           0.63         0.49         0.00           0.67
Avg. external library imports per PR           1.87           0.73           0.08         0.73         1.65           0.83

In [5]:
# RQ3: Choosing Libraries Table
# Languages across the top (consistent with main table)

import pandas as pd

print("\n" + "=" * 80)
print("RQ3: Choosing Libraries")
print("=" * 80)

# Build RQ3 table data
rq3_rows = []

# Row 0: Top 10 external library imports (with counts)
top_imports_data = {}
for lang in langs:
    top_10 = stats_by_language[lang]["top_extlib_imports"][:10]
    # Filter out empty library names and escape underscores
    imports_list = []
    for imp in top_10:
        lib_name = imp["library"].strip()
        if lib_name:  # Skip empty names
            # Escape underscores for LaTeX
            lib_name_escaped = lib_name.replace("_", "\\_")
            imports_list.append(f"{lib_name_escaped} ({imp['count']})")

    # Use \makecell to get newlines
    imports_str = " \\\\ ".join(imports_list)
    top_imports_data[lang] = f"\\makecell[l]{{{imports_str}}}"

rq3_rows.append(
    {
        "Metric": "\\makecell[l]{Top 10 external \\\\ library imports}",
        **top_imports_data,
    }
)

# Row 1: Top 10 new dependencies (with counts)
top_deps_data = {}
for lang in langs:
    top_10 = stats_by_language[lang]["top_new_deps"][:10]
    # Escape underscores for LaTeX
    deps_list = []
    for dep in top_10:
        lib_name = dep["library"].replace("_", "\\_")
        deps_list.append(f"{lib_name} ({dep['count']})")

    # Use \makecell to get newlines
    deps_str = " \\\\ ".join(deps_list)
    top_deps_data[lang] = f"\\makecell[l]{{{deps_str}}}"

rq3_rows.append(
    {
        "Metric": "\\makecell[l]{Top 10 new \\\\ dependencies}",
        **top_deps_data,
    }
)


df_rq3 = pd.DataFrame(rq3_rows)
print("\nPreview of RQ3 data:")
print(df_rq3.to_string(index=False))

# Generate LaTeX table for RQ3
# Metric column left-aligned, library list columns left-aligned
latex_rq3_lines = []
latex_rq3_lines.append("\\begin{tabular}{|l|l|l|l|l|l|}")
latex_rq3_lines.append("\\hline")

# Header row - center-aligned using \multicolumn
header_parts = ["\\textbf{Metric}"]
for label in lang_labels:
    header_parts.append(f"\\multicolumn{{1}}{{c|}}{{\\textbf{{{label}}}}}")
header = " & ".join(header_parts) + " \\\\"
latex_rq3_lines.append(header)
latex_rq3_lines.append("\\hline")

# Row 0: Top 10 imports (text - left-aligned)
latex_rq3_lines.append(
    f"{rq3_rows[0]['Metric']} & {rq3_rows[0]['typescript']} & {rq3_rows[0]['python']} & {rq3_rows[0]['go']} & {rq3_rows[0]['csharp']} & {rq3_rows[0]['rust']} \\\\"
)
latex_rq3_lines.append("\\hline")

# Row 1: Top 10 new deps (text - left-aligned)
latex_rq3_lines.append(
    f"{rq3_rows[1]['Metric']} & {rq3_rows[1]['typescript']} & {rq3_rows[1]['python']} & {rq3_rows[1]['go']} & {rq3_rows[1]['csharp']} & {rq3_rows[1]['rust']} \\\\"
)
latex_rq3_lines.append("\\hline")

latex_rq3_lines.append("\\end{tabular}")

latex_rq3_table = "\n".join(latex_rq3_lines)

print("\n" + "=" * 80)
print("RQ3 LaTeX Table:")
print("=" * 80)
print(latex_rq3_table)


RQ3: Choosing Libraries

Preview of RQ3 data:
                                          Metric                                                                                                                                                                                                                   typescript                                                                                                                                                                              python                                                                                                                                                                                                                                                                                                                                                                     go                                                                                                                                                 

## Save All LaTeX Tables to File

In [4]:
# Save all tables to a single LaTeX file
latex_output = output_dir / "latex_tables.tex"

with open(latex_output, "w") as f:
    f.write("% Generated LaTeX tables for research paper\n")
    f.write("% Generated from aggregated statistics\n")
    f.write(
        "% Requires \\usepackage{multirow} and \\usepackage{makecell} in your LaTeX preamble\n\n"
    )

    f.write("% Combined General + RQ1 + RQ2 Overview\n")
    f.write(latex_combined)
    f.write("\n\n\n")

    # Add RQ3 table
    f.write("% RQ3: Choosing Libraries - Per Language Breakdown\n")
    f.write(latex_rq3_table)
    f.write("\n\n\n")

print(f"Saved all LaTeX tables to {latex_output}")

Saved all LaTeX tables to ../output/latex_tables.tex
